# Unified vs Segment-Specific Comparison

This notebook creates a compact comparison table between:
- the **segment-specific best model** for each market category, and
- the **unified model baseline** from the unified modelling pipeline.

The final table includes:
1. Market Category
2. Optimal Algorithm
3. Coefficient of Determination ($R^2$)
4. RMSE (Segment-Specific vs. Unified)
5. Error Reduction (%)

## 1. Imports and File Resolution

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() or ((p / 'data').exists() and (p / 'reports').exists()):
            return p
        p = p.parent
    return Path.cwd().resolve()

repo_root = find_repo_root()
tables_dir = repo_root / 'reports' / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)

segment_best_path = tables_dir / 'segmentwise_best_model_per_segment.csv'
if not segment_best_path.exists():
    raise FileNotFoundError(f'Missing required file: {segment_best_path}')

# Prefer with_supply_demand outputs when available, else fallback to no_supply_demand.
unified_segment_candidates = [
    #tables_dir / 'unified_segment_evaluation_with_supply_demand.csv',
    tables_dir / 'unified_segment_evaluation_no_supply_demand.csv',
]
unified_summary_candidates = [
    #tables_dir / 'unified_cv_summary_with_supply_demand.csv',
    tables_dir / 'unified_cv_summary_no_supply_demand.csv',
]

unified_segment_path = next((p for p in unified_segment_candidates if p.exists()), None)
unified_summary_path = next((p for p in unified_summary_candidates if p.exists()), None)

if unified_segment_path is None:
    raise FileNotFoundError('Missing unified segment evaluation CSV (with/no supply-demand).')
if unified_summary_path is None:
    raise FileNotFoundError('Missing unified CV summary CSV (with/no supply-demand).')

print(f'Loaded segment-wise best file: {segment_best_path.name}')
print(f'Loaded unified segment file: {unified_segment_path.name}')
print(f'Loaded unified summary file: {unified_summary_path.name}')

Loaded segment-wise best file: segmentwise_best_model_per_segment.csv
Loaded unified segment file: unified_segment_evaluation_no_supply_demand.csv
Loaded unified summary file: unified_cv_summary_no_supply_demand.csv


## 2. Build Requested Comparison Table

In [7]:
segment_best = pd.read_csv(segment_best_path).copy()
unified_seg = pd.read_csv(unified_segment_path).copy()
unified_summary = pd.read_csv(unified_summary_path).copy()

# Normalize segment/category names for robust matching.
def norm_text(s: str) -> str:
    return str(s).strip().lower().replace('-', '_').replace(' ', '_')

segment_best['segment_norm'] = segment_best['Segment'].apply(norm_text)
unified_seg['segment_norm'] = unified_seg['Segment'].apply(norm_text)

# Harmonize unified metric column names across possible exports.
if 'RMSE' in unified_seg.columns and 'RMSE_unified' not in unified_seg.columns:
    unified_seg = unified_seg.rename(columns={'RMSE': 'RMSE_unified'})
if 'R2' in unified_seg.columns and 'R2_unified' not in unified_seg.columns:
    unified_seg = unified_seg.rename(columns={'R2': 'R2_unified'})

required_unified_cols = {'segment_norm', 'RMSE_unified', 'R2_unified'}
missing_unified = required_unified_cols - set(unified_seg.columns)
if missing_unified:
    raise ValueError(f'Unified segment file is missing required columns: {sorted(missing_unified)}')

merged = segment_best.merge(
    unified_seg[['segment_norm', 'RMSE_unified', 'R2_unified']],
    on='segment_norm',
    how='inner',
)

if merged.empty:
    raise ValueError('No matching market categories found between segment-wise and unified outputs.')

# Positive values mean segment-specific reduces error vs unified baseline.
merged['error_reduction_pct'] = np.where(
    merged['RMSE_unified'] > 0,
    (merged['RMSE_unified'] - merged['RMSE_mean']) / merged['RMSE_unified'] * 100.0,
    np.nan,
)

# Requested report table (segment-level rows).
findings_table = pd.DataFrame({
    'Market Category': merged['Segment'],
    'Optimal Algorithm': merged['Model'],
    'Coefficient of Determination (R^2)': merged['R2_mean'],
    'RMSE (Segment-Specific vs. Unified)': merged.apply(
        lambda r: f"{r['RMSE_mean']:.2f} vs {r['RMSE_unified']:.2f}", axis=1
    ),
    'Error Reduction (%)': merged['error_reduction_pct'],
})

# Add one unified baseline row as requested context.
best_unified_row = unified_summary.sort_values('RMSE_mean', ascending=True).iloc[0]
findings_table = pd.concat([
    findings_table,
    pd.DataFrame([{
        'Market Category': 'Unified',
        'Optimal Algorithm': best_unified_row['Model'],
        'Coefficient of Determination (R^2)': best_unified_row['R2_mean'],
        'RMSE (Segment-Specific vs. Unified)': f"{best_unified_row['RMSE_mean']:.2f} vs {best_unified_row['RMSE_mean']:.2f}",
        'Error Reduction (%)': 0.0,
    }]),
], ignore_index=True)

display(findings_table.round({'Coefficient of Determination (R^2)': 4, 'Error Reduction (%)': 2}))

,Market Category,Optimal Algorithm,Coefficient of Determination (R^2),RMSE (Segment-Specific vs. Unified),Error Reduction (%)
0,Dust,Random Forest,0.4151,158.00 vs 52.46,-201.17
1,High Grown,Random Forest,0.2871,224.77 vs 77.65,-189.46
2,Low Grown,Gradient Boosting,0.8581,280.54 vs 49.24,-469.70
3,Off-Grade,XGBoost,0.2832,145.11 vs 48.49,-199.23
4,Unified,Random Forest,0.8728,221.74 vs 221.74,0.00


## 3. Save Table Outputs

In [8]:
suffix = '_with_supply_demand' if 'with_supply_demand' in unified_segment_path.name else '_no_supply_demand'

out_csv = tables_dir / f'unified_vs_segment_key_findings_table{suffix}.csv'
out_md = tables_dir / f'unified_vs_segment_key_findings_table{suffix}.md'

findings_table.to_csv(out_csv, index=False)

def df_to_markdown_pipe(df: pd.DataFrame) -> str:
    cols = [str(c) for c in df.columns]
    lines = [
        '| ' + ' | '.join(cols) + ' |',
        '| ' + ' | '.join(['---'] * len(cols)) + ' |',
    ]
    for _, row in df.iterrows():
        vals = []
        for c in cols:
            v = row[c]
            if pd.isna(v):
                vals.append('')
            else:
                vals.append(str(v).replace('|', '\\|'))
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines)

with open(out_md, 'w', encoding='utf-8') as f:
    f.write(df_to_markdown_pipe(findings_table))

print('Saved outputs:')
print(f'- {out_csv}')
print(f'- {out_md}')

Saved outputs:
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_table_no_supply_demand.csv
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_table_no_supply_demand.md


## 4. Statistical Validation (Fold-Wise)

This section reproduces the rigorous unified-vs-segment evaluation on the base processed dataset using:
- `TimeSeriesSplit` (5 folds)
- paired t-test on fold RMSE
- Diebold-Mariano test on forecast errors

In [9]:
import warnings
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)

TARGET_COL = 'price_mid_lkr'
SEGMENT_COL = 'category_type'
SALE_NO_COL = 'sale_number'
SALE_ID_COL = 'sale_id'
N_CV_FOLDS = 5
RANDOM_STATE = 42

def make_model():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('regressor', RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

if 'with_supply_demand' in unified_segment_path.name:
    stat_candidates = [
        repo_root / 'data' / 'Processed' / 'tea_preprocessed_t13_supply_demand.csv',
        repo_root / 'data' / 'Processed' / 'tea_preprocessed_v2.csv',
    ]
else:
    stat_candidates = [
        repo_root / 'data' / 'Processed' / 'tea_preprocessed_v2.csv',
        repo_root / 'data' / 'Processed' / 'tea_preprocessed.csv',
    ]

stat_data_path = next((p for p in stat_candidates if p.exists()), None)
if stat_data_path is None:
    raise FileNotFoundError('No processed dataset found for statistical validation.')

df_raw = pd.read_csv(stat_data_path)
df = df_raw.loc[df_raw[TARGET_COL].notna()].copy()
segments = sorted(df[SEGMENT_COL].dropna().astype(str).unique())

if 'sale_year' in df.columns:
    df['_sale_year_num'] = pd.to_numeric(df['sale_year'], errors='coerce').fillna(0)
else:
    df['_sale_year_num'] = 0

df['_sale_num'] = pd.to_numeric(df[SALE_NO_COL], errors='coerce').fillna(0)
df = df.sort_values(['_sale_year_num', '_sale_num'], kind='mergesort').reset_index(drop=True)
df['_time_order'] = np.arange(len(df))

EXCLUDE_COLS = {
    SALE_ID_COL, 'sale_date_raw', 'sale_month', 'grade', 'tier',
    SEGMENT_COL, 'table_source', 'elevation', 'category',
    'price_lo_lkr', 'price_hi_lkr', 'price_range_lkr',
    TARGET_COL, 'price_mid_lkr_log', 'has_price_target', 'price_mid_usd',
    '_sale_year_num', '_sale_num', '_time_order',
    '_sale_date_parsed', '_sale_number_numeric', '_sale_order',
    '_current_supply_proxy', '_current_demand_proxy',
    '_historical_supply_avg', '_historical_demand_avg',
}

feature_cols = [
    c for c in df.columns
    if c not in EXCLUDE_COLS and pd.api.types.is_numeric_dtype(df[c])
]

X_all = df[feature_cols].values
y_all = df[TARGET_COL].values
segments_arr = df[SEGMENT_COL].astype(str).values

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    denom = np.where(np.abs(y_true) < 1e-9, np.nan, y_true)
    mape = float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100)
    return {'rmse': rmse, 'mae': mae, 'mape': mape, 'r2': r2}

def diebold_mariano_test(e1: np.ndarray, e2: np.ndarray, horizon: int = 1, power: int = 2) -> tuple[float, float]:
    d = np.abs(e1) ** power - np.abs(e2) ** power
    n = len(d)
    if n < 3:
        return np.nan, np.nan

    d_mean = np.mean(d)
    gamma_0 = np.var(d, ddof=1)
    gamma_sum = 0.0
    for k in range(1, horizon):
        gamma_k = np.cov(d[k:], d[:-k], ddof=1)[0, 1] if n > k else 0.0
        gamma_sum += gamma_k
    var_d = (gamma_0 + 2 * gamma_sum) / n

    if var_d <= 0:
        return np.nan, np.nan

    dm_stat = d_mean / np.sqrt(var_d)
    k = ((n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n) ** 0.5
    dm_stat_corrected = dm_stat * k if k > 0 else dm_stat
    p_value = 2.0 * stats.t.sf(np.abs(dm_stat_corrected), df=n - 1)
    return float(dm_stat_corrected), float(p_value)

tscv = TimeSeriesSplit(n_splits=N_CV_FOLDS)
fold_metrics_rows = []
prediction_rows = []

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_all), start=1):
    X_train_fold, X_test_fold = X_all[train_idx], X_all[test_idx]
    y_train_fold, y_test_fold = y_all[train_idx], y_all[test_idx]
    seg_train_fold = segments_arr[train_idx]
    seg_test_fold = segments_arr[test_idx]

    unified_model = make_model()
    unified_model.fit(X_train_fold, y_train_fold)
    unified_pred = unified_model.predict(X_test_fold)

    segment_pred = np.full(unified_pred.shape, np.nan, dtype=float)
    for seg in segments:
        seg_train_mask = seg_train_fold == seg
        seg_test_mask = seg_test_fold == seg

        n_train_seg = int(seg_train_mask.sum())
        n_test_seg = int(seg_test_mask.sum())

        if n_test_seg == 0:
            continue

        if n_train_seg < 5:
            segment_pred[seg_test_mask] = unified_pred[seg_test_mask]
            continue

        seg_model = make_model()
        seg_model.fit(X_train_fold[seg_train_mask], y_train_fold[seg_train_mask])
        segment_pred[seg_test_mask] = seg_model.predict(X_test_fold[seg_test_mask])

    for i, global_idx in enumerate(test_idx):
        actual = y_test_fold[i]
        u_pred = unified_pred[i]
        s_pred = segment_pred[i]
        seg = seg_test_fold[i]
        prediction_rows.append({
            'fold': fold_idx,
            'global_idx': int(global_idx),
            'segment': seg,
            'actual': float(actual),
            'pred_unified': float(u_pred),
            'pred_segment': float(s_pred) if not np.isnan(s_pred) else np.nan,
            'error_unified': float(actual - u_pred),
            'error_segment': float(actual - s_pred) if not np.isnan(s_pred) else np.nan,
        })

    for seg in segments:
        seg_mask = seg_test_fold == seg
        if int(seg_mask.sum()) == 0:
            continue

        y_seg = y_test_fold[seg_mask]
        u_seg = unified_pred[seg_mask]
        s_seg = segment_pred[seg_mask]

        if np.all(np.isnan(s_seg)):
            continue

        m_uni = compute_metrics(y_seg, u_seg)
        m_seg = compute_metrics(y_seg, s_seg)

        fold_metrics_rows.append({
            'fold': fold_idx,
            'segment': seg,
            'n_test': int(seg_mask.sum()),
            'rmse_unified': m_uni['rmse'],
            'mae_unified': m_uni['mae'],
            'mape_unified': m_uni['mape'],
            'r2_unified': m_uni['r2'],
            'rmse_segment': m_seg['rmse'],
            'mae_segment': m_seg['mae'],
            'mape_segment': m_seg['mape'],
            'r2_segment': m_seg['r2'],
        })

df_fold_metrics = pd.DataFrame(fold_metrics_rows)
df_predictions = pd.DataFrame(prediction_rows)

summary_rows = []
for seg in segments:
    seg_folds = df_fold_metrics[df_fold_metrics['segment'] == seg]
    seg_preds = df_predictions[df_predictions['segment'] == seg]

    if len(seg_folds) < 2:
        continue

    rmse_u = seg_folds['rmse_unified'].values
    rmse_s = seg_folds['rmse_segment'].values
    r2_u = seg_folds['r2_unified'].values
    r2_s = seg_folds['r2_segment'].values

    rmse_u_mean = float(np.mean(rmse_u))
    rmse_s_mean = float(np.mean(rmse_s))
    pct_imp_rmse = ((rmse_u_mean - rmse_s_mean) / rmse_u_mean * 100.0) if rmse_u_mean > 0 else np.nan

    _, t_pval = stats.ttest_rel(rmse_u, rmse_s) if len(rmse_u) >= 2 else (np.nan, np.nan)

    e_uni = seg_preds['error_unified'].dropna().values
    e_seg = seg_preds['error_segment'].dropna().values
    min_len = min(len(e_uni), len(e_seg))
    _, dm_pval = diebold_mariano_test(e_uni[:min_len], e_seg[:min_len]) if min_len >= 5 else (np.nan, np.nan)

    if rmse_s_mean < rmse_u_mean:
        better = 'segment-specific'
    elif rmse_s_mean > rmse_u_mean:
        better = 'unified'
    else:
        better = 'tie'

    summary_rows.append({
        'segment': seg,
        'n_folds': int(len(seg_folds)),
        'n_obs': int(len(seg_preds)),
        'rmse_unified_mean': round(rmse_u_mean, 2),
        'rmse_segment_mean': round(rmse_s_mean, 2),
        'pct_improvement_rmse': round(pct_imp_rmse, 2),
        'r2_unified_mean': round(float(np.mean(r2_u)), 4),
        'r2_segment_mean': round(float(np.mean(r2_s)), 4),
        'paired_t_pvalue': round(float(t_pval), 6) if pd.notna(t_pval) else np.nan,
        'dm_pvalue': round(float(dm_pval), 6) if pd.notna(dm_pval) else np.nan,
        'better_model': better,
    })

df_stat_summary = pd.DataFrame(summary_rows).sort_values('segment').reset_index(drop=True)

display(df_stat_summary)
print(f'Statistical validation source data: {stat_data_path.name}')
print(f'Features used: {len(feature_cols)}')

,segment,n_folds,n_obs,rmse_unified_mean,rmse_segment_mean,pct_improvement_rmse,r2_unified_mean,r2_segment_mean,paired_t_pvalue,dm_pvalue,better_model
0,dust,5,421,150.26,153.72,-2.30,0.4643,0.4314,0.426659,0.184213,unified
1,high_grown,5,420,251.04,251.13,-0.03,0.1134,0.1069,0.981806,0.845920,unified
2,low_grown,5,1138,258.17,259.26,-0.42,0.8683,0.8718,0.907528,0.509410,unified
3,off_grade,5,426,143.73,144.29,-0.39,0.2891,0.2857,0.664095,0.594195,unified


Statistical validation source data: tea_preprocessed_v2.csv
Features used: 228


## 5. Save Statistical Validation Outputs

In [10]:
stat_fold_csv = tables_dir / f'unified_vs_segment_key_findings_fold_metrics{suffix}.csv'
stat_pred_csv = tables_dir / f'unified_vs_segment_key_findings_predictions{suffix}.csv'
stat_summary_csv = tables_dir / f'unified_vs_segment_key_findings_stat_summary{suffix}.csv'

df_fold_metrics.to_csv(stat_fold_csv, index=False)
df_predictions.to_csv(stat_pred_csv, index=False)
df_stat_summary.to_csv(stat_summary_csv, index=False)

print('Saved statistical outputs:')
print(f'- {stat_fold_csv}')
print(f'- {stat_pred_csv}')
print(f'- {stat_summary_csv}')

Saved statistical outputs:
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_fold_metrics_no_supply_demand.csv
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_predictions_no_supply_demand.csv
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_stat_summary_no_supply_demand.csv
